# Telco Customer Churn SQL Analysis Notebook

This notebook loads the cleaned Telco churn dataset into an in-memory SQLite database and runs the SQL queries used for the descriptive analysis

using `Datasets/cleaned_telco_churn_for_sql.csv` 

In [2]:
import pandas as pd
import sqlite3
from pathlib import Path

# Find the cleaned SQL dataset
possible_paths = [
    Path("Datasets/cleaned_telco_churn_for_sql.csv"),

]

for path in possible_paths:
    if path.exists():
        DATA_PATH = path
        break
else:
    raise FileNotFoundError(
        "Could not find cleaned_telco_churn_for_sql.csv. "
        "Place it in the Datasets folder or update DATA_PATH manually."
    )

print(f"Loading data from: {DATA_PATH.resolve()}")

# Load data
df = pd.read_csv(DATA_PATH)

# Create SQLite in-memory database
conn = sqlite3.connect(":memory:")
df.to_sql("telco_churn", conn, index=False, if_exists="replace")

print(f"Rows loaded: {df.shape[0]:,}")
print(f"Columns loaded: {df.shape[1]:,}")

def run_sql(query):
    """Run a SQL query against the telco_churn SQLite table."""
    return pd.read_sql_query(query, conn)

# Preview the dataset
df.head()

Loading data from: C:\Users\Meklit\Desktop\AI data analytics projects\Project 1 Telco Customer Churn\1. Customer_churn\Datasets\cleaned_telco_churn_for_sql.csv
Rows loaded: 7,043
Columns loaded: 28


,CustomerID,City,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Paperless Billing,Monthly Charges,...,Tech Support_Yes,Streaming TV_Yes,Streaming Movies_Yes,Contract_One year,Contract_Two year,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check,Payment Risk Category,Tenure Group
0,3668-QPYBK,Los Angeles,1,0,0,0,2,1,1,53.85,...,0,0,0,0,0,0,0,1,Medium Risk,New
1,9237-HQITU,Los Angeles,0,0,0,1,2,1,1,70.70,...,0,0,0,0,0,0,1,0,High Risk,New
2,9305-CDSKC,Los Angeles,0,0,0,1,8,1,1,99.65,...,0,1,1,0,0,0,1,0,High Risk,New
3,7892-POOKP,Los Angeles,0,0,1,1,28,1,1,104.80,...,1,1,1,0,0,0,1,0,High Risk,Long term
4,0280-XJGEX,Los Angeles,1,0,0,1,49,1,1,103.70,...,0,1,1,0,0,0,0,0,Low Risk,Long term


## Query Dictionary

The queries below are stored in a Python dictionary so they are easy to run in this notebook and easy to reuse later in Streamlit.


In [3]:
queries = {
    "overall_churn_rate": """
SELECT
    COUNT("CustomerID") AS total_customers,
    SUM("Churn Value") AS total_churned,
    ROUND((SUM("Churn Value") * 1.0 / NULLIF(COUNT("CustomerID"), 0)) * 100, 2) AS churn_rate_percentage
FROM telco_churn;
""",

    "approx_monthly_churn_rate": """
SELECT
    SUM("Churn Value") AS total_churned,
    SUM("Tenure Months") AS total_customer_months,
    ROUND((SUM("Churn Value") * 1.0 / NULLIF(SUM("Tenure Months"), 0)) * 100, 2) AS avg_monthly_churn_rate_percentage
FROM telco_churn;
""",

    "churn_by_contract_type": """
SELECT
    CASE
        WHEN "Contract_One year" = 1 THEN 'One Year Contract'
        WHEN "Contract_Two year" = 1 THEN 'Two Year Contract'
        ELSE 'Month-to-Month Contract'
    END AS contract_type,
    SUM("Churn Value") AS total_churned,
    COUNT("CustomerID") AS total_customers,
    ROUND((SUM("Churn Value") * 1.0 / NULLIF(COUNT("CustomerID"), 0)) * 100, 2) AS churn_rate_percentage
FROM telco_churn
GROUP BY 1
ORDER BY churn_rate_percentage DESC;
""",

    "churn_by_internet_type": """
SELECT
    CASE
        WHEN "Internet Service_Fiber optic" = 1 THEN 'Fiber Optic'
        WHEN "Internet Service_No" = 1 THEN 'No Internet Service'
        ELSE 'DSL'
    END AS internet_type,
    SUM("Churn Value") AS total_churned,
    COUNT("CustomerID") AS total_customers,
    ROUND((SUM("Churn Value") * 1.0 / NULLIF(COUNT("CustomerID"), 0)) * 100, 2) AS churn_rate_percentage
FROM telco_churn
GROUP BY 1
ORDER BY churn_rate_percentage DESC;
""",

    "high_value_customer_churn_rate": """
WITH ThresholdCalc AS (
    SELECT "Monthly Charges" AS cutoff_value
    FROM telco_churn
    ORDER BY "Monthly Charges"
    LIMIT 1 OFFSET (SELECT CAST(COUNT(*) * 0.75 AS INTEGER) FROM telco_churn)
)
SELECT
    CASE
        WHEN "Monthly Charges" >= (SELECT cutoff_value FROM ThresholdCalc) THEN 'High Value'
        ELSE 'Other'
    END AS customer_segment,
    COUNT("CustomerID") AS total_customers,
    SUM("Churn Value") AS total_churned,
    ROUND((SUM("Churn Value") * 1.0 / NULLIF(COUNT("CustomerID"), 0)) * 100, 2) AS churn_rate_percentage
FROM telco_churn
GROUP BY 1
ORDER BY churn_rate_percentage DESC;
""",

    "high_value_customer_summary": """
WITH ThresholdCalc AS (
    SELECT "Monthly Charges" AS cutoff_value
    FROM telco_churn
    ORDER BY "Monthly Charges"
    LIMIT 1 OFFSET (SELECT CAST(COUNT(*) * 0.75 AS INTEGER) FROM telco_churn)
)
SELECT
    COUNT("CustomerID") AS total_high_value_customers,
    SUM("Churn Value") AS total_high_value_churned,
    ROUND((SUM("Churn Value") * 1.0 / NULLIF(COUNT("CustomerID"), 0)) * 100, 2) AS high_value_churn_rate_percentage
FROM telco_churn
WHERE "Monthly Charges" >= (SELECT cutoff_value FROM ThresholdCalc);
""",

    "average_revenue_per_month": """
SELECT
    ROUND(AVG("Monthly Charges"), 2) AS arpu
FROM telco_churn;
""",

    "historical_average_revenue_per_month": """
SELECT
    ROUND(SUM("Total Charges") / NULLIF(SUM("Tenure Months"), 0), 2) AS historical_arpu
FROM telco_churn;
""",

    "estimated_customer_lifetime_value": """
SELECT
    ROUND(AVG("Monthly Charges") * AVG("Tenure Months"), 2) AS estimated_avg_cltv
FROM telco_churn;
""",

    "historical_customer_lifetime_value": """
SELECT
    ROUND(AVG("Total Charges"), 2) AS historical_avg_cltv
FROM telco_churn;
""",

    "churn_by_tenure": """
WITH ChurnByTenure AS (
    SELECT
        "Tenure Months",
        SUM("Churn Value") AS churned_at_tenure
    FROM telco_churn
    GROUP BY "Tenure Months"
)
SELECT
    "Tenure Months",
    churned_at_tenure,
    SUM(churned_at_tenure) OVER (ORDER BY "Tenure Months") AS cumulative_churned,
    ROUND((SUM(churned_at_tenure) OVER (ORDER BY "Tenure Months") * 1.0 /
           (SELECT COUNT("CustomerID") FROM telco_churn)) * 100, 2) AS true_running_churn_percentage
FROM ChurnByTenure
ORDER BY "Tenure Months";
""",

    "high_risk_customers_ranked": """
SELECT
    "CustomerID",
    "Monthly Charges",
    "Payment Risk Category",
    RANK() OVER(ORDER BY "Monthly Charges" DESC) AS monthly_charges_rank
FROM telco_churn
WHERE "Payment Risk Category" = 'High Risk'
ORDER BY monthly_charges_rank
LIMIT 25;
"""
}

## Overall Churn Rate

Calculated as total churned customers divided by total customers to measure the overall percentage of customers who left Telco.

In [4]:
run_sql(queries["overall_churn_rate"])

,total_customers,total_churned,churn_rate_percentage
0,7043,1869,26.54


## Approximate Monthly Churn Rate
Calculated as total churned customers divided by total customer tenure months to estimate churn per customer-month because the dataset is a single snapshot.

In [5]:
run_sql(queries["approx_monthly_churn_rate"])

,total_churned,total_customer_months,avg_monthly_churn_rate_percentage
0,1869,227990,0.82


## Churn by Contract Type

Calculated by grouping customers by contract type and dividing churned customers by total customers in each contract category to compare churn risk across contract lengths.

In [6]:
run_sql(queries["churn_by_contract_type"])

,contract_type,total_churned,total_customers,churn_rate_percentage
0,Month-to-Month Contract,1655,3875,42.71
1,One Year Contract,166,1473,11.27
2,Two Year Contract,48,1695,2.83


## Churn by Internet Type
Calculated by grouping customers by internet service type and dividing churned customers by total customers in each group to identify which service category has the highest churn risk

In [7]:
run_sql(queries["churn_by_internet_type"])

,internet_type,total_churned,total_customers,churn_rate_percentage
0,Fiber Optic,1297,3096,41.89
1,DSL,459,2421,18.96
2,No Internet Service,113,1526,7.40


## High Value Customer Churn Rate
Calculated by identifying customers in the top 25% of monthly charges and comparing their churn rate to all other customers to assess whether high-spending customers are at greater risk.

In [8]:
run_sql(queries["high_value_customer_churn_rate"])

,customer_segment,total_customers,total_churned,churn_rate_percentage
0,High Value,1771,580,32.75
1,Other,5272,1289,24.45


## High-Value Customer Summary

In [9]:
run_sql(queries["high_value_customer_summary"])

,total_high_value_customers,total_high_value_churned,high_value_churn_rate_percentage
0,1771,580,32.75


## Average Revenue per Month
Calculated using the average monthly charge per customer to estimate Telco’s typical monthly revenue per user (using monthly charges column)

In [10]:
run_sql(queries["average_revenue_per_month"])

,arpu
0,64.76


## Historical Average Revenue per Month

In [11]:
run_sql(queries["historical_average_revenue_per_month"])

,historical_arpu
0,70.42


## Estimated Customer Lifetime Value
Calculated as average monthly charges multiplied by average tenure months to estimate the average revenue generated over a customer’s lifetime.

In [12]:
run_sql(queries["estimated_customer_lifetime_value"])

,estimated_avg_cltv
0,2096.41


## Historical Customer Lifetime Value
Calculated using the average total charges per customer to estimate lifetime value based on actual accumulated customer spending.

In [13]:
run_sql(queries["historical_customer_lifetime_value"])


,historical_avg_cltv
0,2279.73


## Churn by Tenure

Calculated by grouping churned customers by tenure month to analyze how churn changes as customers remain with Telco longer.

In [14]:
run_sql(queries["churn_by_tenure"])

,Tenure Months,churned_at_tenure,cumulative_churned,true_running_churn_percentage
0,0,0,0,0.00
1,1,380,380,5.40
2,2,123,503,7.14
3,3,94,597,8.48
4,4,83,680,9.65
...,...,...,...,...
68,68,9,1838,26.10
69,69,8,1846,26.21
70,70,11,1857,26.37
71,71,6,1863,26.45


## High-Risk Customers Ranked
Calculated by ranking high-risk customers by monthly charges to prioritize valuable customers who may need targeted retention efforts.

In [15]:
run_sql(queries["high_risk_customers_ranked"])

,CustomerID,Monthly Charges,Payment Risk Category,monthly_charges_rank
0,8984-HPEMB,118.65,High Risk,1
1,5734-EJKXG,118.60,High Risk,2
2,9924-JPRMC,118.20,High Risk,3
3,2302-ANTDP,117.45,High Risk,4
4,8016-NCFVO,116.50,High Risk,5
5,8879-XUAHX,116.25,High Risk,6
6,4868-AADLV,116.25,High Risk,6
7,0106-UGRDO,116.00,High Risk,8
8,9659-QEQSY,115.65,High Risk,9
9,8263-QMNTJ,115.55,High Risk,10
